<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_04_battery_arbitrage_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **complete** version: every cell is written out and runs as it stands. Read it, run it, and check what you see against the note under each section.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Prince, *Understanding Deep Learning*, MIT Press 2023.
- Goodfellow, Bengio & Courville, *Deep Learning*, MIT Press 2016.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_06 · Notebook 04 — A Battery That Learns to Trade

**Paired with L6.2 · Training Philosophies**

A 200 kWh battery trades on the day-ahead market: each hour it buys at full
power, waits, or sells, and the only feedback is money. That is a reinforcement
learning problem. It can also be solved **exactly** by a linear program, which
makes it a fair test of how close reinforcement learning gets, and of how much
closer a network gets by copying the optimiser. You will

1. build the battery: state, action, reward;
2. solve each day exactly with a linear program, the benchmark;
3. try a simple rule: buy in the two cheapest hours, sell in the two dearest;
4. train a policy by REINFORCE, the policy gradient of L6.2;
5. compare the three;
6. train a network to imitate the linear program: approximate MPC;
7. time the optimiser against the network.

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['Ex_6_core.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex06-training-lab/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))

In [ ]:
# outputs-cell v1 --------------------------------------------------------
# Later notebooks in this set read results that earlier ones save. On Google
# Colab every notebook runs on its own temporary machine, so a file saved
# here is not there when the next notebook opens. This cell keeps the
# results in your Google Drive instead: approve the access request when it
# appears. If you decline it, or have no Google Drive, the results are
# downloaded to your computer when saved and the notebook that needs them
# asks for them back. Locally this cell does nothing.
import Ex_6_core as core
core.keep_outputs()

In [ ]:
import os
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

import Ex_6_core as core

core.set_seed(0)
E_MAX, P_MAX, ETA, C_DEG, SOC0 = (core.BATTERY[k] for k in ("E_MAX", "P_MAX", "ETA", "C_DEG", "SOC0"))
T = core.HOURS                                                  # 24 hourly steps

P_TRAIN = torch.tensor(core.arbitrage_prices(400, seed=1))     # 400 days to learn on
P_TEST = torch.tensor(core.arbitrage_prices(50, seed=2))       # 50 days never trained on
print("training days:", tuple(P_TRAIN.shape), "   test days:", tuple(P_TEST.shape))
print(f"test-day prices from {P_TEST.min():.3f} to {P_TEST.max():.3f} EUR/kWh")

fig, ax = plt.subplots(figsize=(7.2, 3.6))
for d in range(3):
    ax.plot(range(T), P_TEST[d].numpy(), lw=1.8, label=f"test day {d}")
ax.set_xlabel("hour"); ax.set_ylabel("price, EUR/kWh")
ax.legend(frameon=False); ax.grid(alpha=0.25)
plt.show()

**What you should see.** `training days: (400, 24)`, `test days: (50, 24)`,
prices from about 0.01 to 0.42 EUR/kWh, and three curves with a
morning peak, a dip at noon where solar pushes the price down, and a higher
evening peak.

The battery holds 200 kWh and moves at most 50 kW, so it takes four hours to
fill. It loses 5 % each way, and every kWh through it costs 0.02 EUR of ageing.
It starts the day half full and must end at least half full; each kWh short is
charged at 0.30 EUR.

In the language of L6.2:

| | here | a control engineer would say |
| --- | --- | --- |
| state *s* | charge, hour, price now and ahead | the state |
| action *a* | buy, wait or sell, 50 kW | the control input |
| policy π(a \| s) | a network | the controller |
| reward *r* | revenue minus ageing, each hour | minus the cost |
| return *G* | the day's profit | the objective |

---

## 1 · The battery

In [ ]:
# the battery, one day at a time --------------------------------------------
def rollout(prices, act, stochastic=False):
    '''Run the battery through a batch of days.

    prices : (B, 24) tensor, EUR/kWh
    act(features, t) returns, for every day in the batch,
        logits over (buy, wait, sell)   when stochastic=True   (the RL policy)
        grid power in kW, + selling     otherwise              (LP, rule, imitation)
    returns rewards (B, 24), log-probabilities (B, 24) or None, features (B, 24, 7)
    '''
    n = prices.shape[0]
    soc = torch.full((n,), SOC0, dtype=prices.dtype)            # kWh stored
    rewards, logps, feats = [], [], []
    for t in range(T):
        f = core.battery_features(soc, t, prices)                # what the policy sees
        out = act(f, t)
        if stochastic:
            dist = torch.distributions.Categorical(logits=out)
            a = dist.sample()                                    # 0 buy, 1 wait, 2 sell
            logps.append(dist.log_prob(a))
            power = (a.to(prices.dtype) - 1.0) * P_MAX           # kW, + selling
        else:
            power = out.to(prices.dtype)
        d = torch.clamp(power, 0, P_MAX)                         # sold this hour, kWh
        c = torch.clamp(-power, 0, P_MAX)                        # bought this hour, kWh
        d = torch.minimum(d, soc * ETA)                          # cannot sell what is not stored
        c = torch.minimum(c, (E_MAX - soc) / ETA)                # cannot store more than there is room for
        soc = soc + ETA * c - d / ETA
        r = prices[:, t] * (d - c) - C_DEG * (c + d)
        if t == T - 1:                                           # end at least where it started
            r = r - core.SHORTFALL_PRICE * torch.clamp(SOC0 - soc, min=0)
        rewards.append(r)
        feats.append(f)
    R = torch.stack(rewards, 1)
    return R, (torch.stack(logps, 1) if logps else None), torch.stack(feats, 1)

# ------------------------------------------------------------------------------
wait = lambda f, t: torch.zeros(len(f))                         # never trade
R_wait, _, _ = rollout(P_TEST, wait)
print(f"never trading : {R_wait.sum(1).mean():6.2f} EUR per day")

**What you should see.** `never trading :   0.00 EUR per day`. A battery that
does nothing earns nothing and ages not at all, which is the first check that
the reward is right.

`rollout` is the simulator, and it is all reinforcement learning is allowed to
know about the battery. It does not see the equations inside, only what comes
out: a reward each hour. That is the point of the method, and in section 5 it
will also be its weakness.

---

## 2 · The exact answer: a linear program

In [ ]:
# The battery is linear and the day's prices are known a day ahead, so the best
# schedule is a linear program: core.lp_schedule solves one day with scipy.
t0 = time.perf_counter()
lp = [core.lp_schedule(p) for p in P_TEST.numpy()]
lp_seconds = (time.perf_counter() - t0) / len(lp)
lp_profit = np.array([x[0] for x in lp])
LP_POWER = torch.tensor(np.stack([x[1] for x in lp]))           # (50, 24) kW, + selling

R_lp, _, _ = rollout(P_TEST, lambda f, t: LP_POWER[:, t])       # the same schedule, through the battery
print(f"linear program            : {lp_profit.mean():6.2f} EUR per day   ({lp_seconds * 1e3:.1f} ms a day)")
print(f"its schedule, in rollout  : {R_lp.sum(1).mean():6.2f} EUR per day")

core.plot_battery_day(P_TEST[0].numpy(), LP_POWER[0].numpy(), title="test day 0: the optimum")
plt.show()

**What you should see.** The linear program earns about **33.15 EUR per day**
on the test days, in about 8 ms a day, and its schedule run through
`rollout` earns the same to the cent — the second check, that the simulator
and the optimiser describe the same battery.

The plot is the shape of every good day: buy in the night and at the noon dip
(orange bars below zero), sell into the morning and evening peaks (green bars
above). This is the best any policy can do, because it was computed with the
whole day's prices and the battery's exact model.

---

## 3 · A simple rule

In [ ]:
def rule_power(prices, hours=2):
    '''Buy at full power in the `hours` cheapest hours, sell in the dearest.'''
    order = prices.argsort(1)
    power = torch.zeros_like(prices)
    power.scatter_(1, order[:, :hours], -P_MAX)                  # the cheapest hours: buy
    power.scatter_(1, order[:, -hours:], P_MAX)                  # the dearest hours: sell
    return power

RULE = rule_power(P_TEST)
R_rule, _, _ = rollout(P_TEST, lambda f, t: RULE[:, t])
print(f"rule                      : {R_rule.sum(1).mean():6.2f} EUR per day")

**What you should see.** About **18.58 EUR per day**: 56 % of the
optimum from one line of reasoning. Two hours, because a half-full battery has
room for two hours of buying; try `hours=4` and watch it empty the battery and
pay the shortfall. Any learned policy has to beat this rule to be worth its
training, and the rule's weakness is plain: it trades once a day, where the
optimum trades twice, around the morning and the evening peaks.

---

## 4 · Reinforcement learning: REINFORCE

The policy is a small network: the seven features in, three logits out (buy,
wait, sell). It plays 128 random training days at a time, choosing its
actions by sampling, and after each batch it raises the probability of the
actions that were followed by a better-than-average return. That is the policy
gradient of L6.2, with the average return as the baseline.

The day is only 24 steps long, so the return is the plain sum of the rewards to
the end of the day; no discount is needed.

In [ ]:
# and TODO 3 --- the return, and the policy-gradient loss
torch.manual_seed(0)
policy = nn.Sequential(nn.Linear(7, 64), nn.Tanh(),              # seven features in
                       nn.Linear(64, 64), nn.Tanh(),
                       nn.Linear(64, 3))                         # logits: buy, wait, sell
opt = torch.optim.Adam(policy.parameters(), lr=3e-3)
greedy = lambda f, t: (policy(f).argmax(1).to(torch.float64) - 1.0) * P_MAX   # its best guess, no sampling

curve, t0 = [], time.time()
for it in range(601):
    days = P_TRAIN[torch.randint(0, len(P_TRAIN), (128,))]      # 128 random training days
    R, logp, _ = rollout(days, lambda f, t: policy(f), stochastic=True)
    G = torch.flip(torch.cumsum(torch.flip(R, [1]), 1), [1])
    baseline = G.mean(0, keepdim=True)                           # the average return, hour by hour
    loss = -(logp * (G - baseline).float()).mean()
    opt.zero_grad(); loss.backward(); opt.step()
    if it % 50 == 0:
        with torch.no_grad():
            curve.append((it, rollout(P_TEST, greedy)[0].sum(1).mean().item()))
        print(f"iteration {it:3d}   test profit {curve[-1][1]:6.2f} EUR per day   ({time.time() - t0:.0f} s)")
# ------------------------------------------------------------------------------

**What you should see.** The test profit starts near 5 EUR per day,
the untrained network trading at random, climbs past the rule within a
hundred iterations or so, and ends near **26.17 EUR per day** after about
10 seconds. The numbers move with the seed; that is sampling, and
L6.2 calls it the variance of the policy gradient.

In [ ]:
with torch.no_grad():
    R_rl, _, _ = rollout(P_TEST, greedy)

its, prof = zip(*curve)
fig, ax = plt.subplots(figsize=(7.2, 3.8))
ax.plot(its, prof, "o-", color="#7d5ba6", lw=1.8, label="REINFORCE, test days")
ax.axhline(lp_profit.mean(), color="#2e9e5b", ls="--", label="linear program")
ax.axhline(R_rule.sum(1).mean().item(), color="#d9822b", ls=":", label="rule")
ax.set_xlabel("iteration"); ax.set_ylabel("profit, EUR per day")
ax.legend(frameon=False); ax.grid(alpha=0.25)
plt.show()

with torch.no_grad():
    soc, powers = torch.tensor([SOC0], dtype=torch.float64), []
    for t in range(T):
        pw = greedy(core.battery_features(soc, t, P_TEST[:1]), t)
        d = torch.clamp(pw, 0, P_MAX).minimum(soc * ETA); c = torch.clamp(-pw, 0, P_MAX).minimum((E_MAX - soc) / ETA)
        soc = soc + ETA * c - d / ETA
        powers.append((d - c).item())
core.plot_battery_day(P_TEST[0].numpy(), np.array(powers), title="test day 0: the learned policy")
plt.show()

---

## 5 · The three, side by side

In [ ]:
rows = []
for name, profit in [("linear program", lp_profit),
                     ("rule", R_rule.sum(1).numpy()),
                     ("REINFORCE", R_rl.sum(1).numpy())]:
    rows.append([name, f"{profit.mean():.2f}", f"{100 * profit.mean() / lp_profit.mean():.0f} %"])
print(core.error_table(rows, ["policy", "EUR per day", "of the optimum"]))

**What you should see.**

| policy | EUR per day | of the optimum |
| --- | --- | --- |
| linear program | 33.15 | 100 % |
| rule | 18.58 | 56 % |
| REINFORCE | 26.17 | 79 % |

Reinforcement learning learned a respectable trader from the reward alone,
without the battery's equations, but stops well short of the optimum after
thousands of simulated days. This problem has a known linear model, so it
should be solved, not learned. Reinforcement learning earns its place when the
model is missing or too hard to optimise.

---

## 6 · Imitation: approximate MPC

The linear program is the expert. Run it offline on 300 training days, record
the state each hour and the power the optimiser chose, and fit a network to
predict the one from the other: ordinary supervised regression, with no reward
anywhere.

In [ ]:
# imitation: regression onto the expert's actions ---------------------------
lp_train = [core.lp_schedule(p) for p in P_TRAIN[:300].numpy()]
POW = torch.tensor(np.stack([x[1] for x in lp_train]))          # the expert's power, (300, 24) kW
_, _, F = rollout(P_TRAIN[:300], lambda f, t: POW[:, t])        # the states the expert visits
X_im = F.reshape(-1, 7)                                          # 7200 states
Y_im = (POW / P_MAX).reshape(-1, 1).float()                      # and actions, scaled to [-1, 1]

torch.manual_seed(0)
imitator = nn.Sequential(nn.Linear(7, 64), nn.Tanh(),
                         nn.Linear(64, 64), nn.Tanh(),
                         nn.Linear(64, 1), nn.Tanh())            # a power between -1 and 1
opt = torch.optim.Adam(imitator.parameters(), lr=3e-3)
for epoch in range(1500):
    loss = nn.functional.mse_loss(imitator(X_im), Y_im)
    opt.zero_grad(); loss.backward(); opt.step()
print(f"imitation loss after training: {loss.item():.4f}")

with torch.no_grad():
    R_im, _, _ = rollout(P_TEST, lambda f, t: imitator(f)[:, 0].double() * P_MAX)
print(f"imitation of the LP       : {R_im.sum(1).mean():6.2f} EUR per day   "
      f"({100 * R_im.sum(1).mean().item() / lp_profit.mean():.0f} % of the optimum)")
# ------------------------------------------------------------------------------

**What you should see.** A training loss of about 0.027, and **31.18 EUR
per day** on the test days, 94 % of the optimum: far closer than
reinforcement learning, from 300 days of examples instead of thousands of
days of trial and error, and with no exploring at all. The network never runs
the battery while it learns, which is why imitation is safe on real hardware.

---

## 7 · The optimiser against the network

In [ ]:
f_day = X_im[:T]                                                 # one day's 24 states
with torch.no_grad():
    t0 = time.perf_counter()
    for _ in range(1000):
        imitator(f_day)
    net_seconds = (time.perf_counter() - t0) / 1000
print(f"linear program : {lp_seconds * 1e3:7.3f} ms per day")
print(f"imitation net  : {net_seconds * 1e3:7.3f} ms per day   ({lp_seconds / net_seconds:.0f} x faster)")

**What you should see.** The linear program in about 8 ms a day and the network
in about 0.05 ms, roughly **160 times faster**; the ratio depends on your
machine, the order of magnitude does not. A battery that trades hourly can
afford the optimiser. A converter whose MPC must finish in fifty microseconds
cannot, and that is where approximate MPC is used.

---

## 8 · What this notebook does not show

* **Perfect foresight.** Every policy was given the whole day's prices; a real
  trader works from a forecast.
* **An exact model.** The battery in `rollout` is exactly the battery in the
  linear program. On hardware it is not, and that is where reinforcement
  learning on a good simulator starts to pay.
* **Compounding error.** The imitator was trained on the states the expert
  visits; on an unusual day it can drift where the expert never went.

---

## 9 · Save

In [ ]:
# Saved for the report in notebook 05.
os.makedirs(core.OUTPUT_DIR, exist_ok=True)
path = os.path.join(core.OUTPUT_DIR, "nb04_arbitrage.npz")
np.savez(path,
         lp=lp_profit, rule=R_rule.sum(1).numpy(), rl=R_rl.sum(1).numpy(),
         imitation=R_im.sum(1).numpy(), curve=np.asarray(curve, dtype=float),
         lp_ms=lp_seconds * 1e3, net_ms=net_seconds * 1e3)
print("wrote", path)
core.saved(path)

**What you should see.** `wrote .../Ex06_outputs/nb04_arbitrage.npz`.

---

## 10 · Before you move on

Answer these here. Each question builds part of an answer to one of the lecture's questions for the oral examination; the arrow under it says which, and the Questions slide at the end of the lecture has them in full.

1. Name the state, the action and the reward of this battery, and say which of
   them a control engineer would call the controller and which the cost. Why
   is the policy judged on the day's profit rather than on each hour's reward?
   *→ L6.2 Q6, Q7*
2. The REINFORCE loss multiplies the log-probability of each action by the
   return that followed, minus the average return. What does one step of
   gradient descent on it do to an action followed by a better-than-average
   day, and why does subtracting the average make training less noisy?
   *→ L6.2 Q8*
3. The linear program earned more than the learned policy, instantly and
   without training. Why could it, and what does that say about when
   reinforcement learning is the wrong tool? Name one change to this problem
   that would make reinforcement learning worth its cost.
   *→ L6.2 Q9*
4. The imitator came much closer to the optimum than REINFORCE and runs far
   faster than the optimiser. What did it need that REINFORCE did not, and
   what would you expect on a day whose prices look nothing like the training
   days?
   *→ L6.2 Q10*

*Write your answers here. You will copy them into the report in notebook 05, which adds them to what you submit.*

1.
2.
3.
4.

---

Next: **[notebook 05](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex06-training-lab/Ex06_05_report.ipynb)**, the report.